# ARG-based inference

When a per-site local tree is available, {class}`~ancestree.inference.ARGBasedInference` infers the ancestral state at each variant directly from its own genealogy. This guide follows a full tree-sequence run and shows why the mutation rate must match the simulator's.

## Loading the tree sequence

We reuse the quickstart dataset: 8 samples (6 ingroup + 2 outgroup), 29 local trees, 224 segregating sites, simulated under a per-generation mutation rate of `5e-8`.

In [1]:
import tskit

ts = tskit.load("quickstart.trees")
print(f"{ts.num_samples} samples, {ts.num_trees} local trees, {ts.num_sites} sites")


8 samples, 29 local trees, 224 sites


## Running the inference

{meth}`Inference.from_arg() <ancestree.inference.Inference.from_arg>` returns an {class}`~ancestree.inference.ARGBasedInference`, whose {meth}`infer() <ancestree.inference.Inference.infer>` yields a ({class}`~ancestree.sites.Site`, {class}`~ancestree.posterior.Posterior`) pair per variant. The ingroup and outgroup names place the reporting node at the ingroup MRCA, and the result is graded on the sites polymorphic within the ingroup against the true allele at that node, as in the {doc}`quickstart`.


In [2]:
import ancestree as anc

ingroup = [f"i{i}" for i in range(6)]
outgroup = ["o0", "o1"]

inf = anc.Inference.from_arg(
    ts, anc.JC69(), mu=5e-8,
    ingroup_samples=ingroup, outgroup_samples=outgroup,
)
res = list(inf.infer())


INFO:ancestree.ARGBasedInference: Inferring the ancestral allele at 224 sites over 29 local trees from the ARG
INFO:ancestree.ARGBasedInference: Using 6 ingroup sample(s): i0, i1, i2, i3, i4, i5
INFO:ancestree.ARGBasedInference: Using 2 outgroup sample(s): o0, o1
ARGBasedInference: 100%|██████████| 29/29 [00:02<00:00, 11.58 trees/s]
INFO:ancestree.ARGBasedInference: Focal node: reported at the ingroup_mrca; 4 tree(s) where the ingroup is not monophyletic, so its MRCA subtends outgroup tips


In [3]:
keep = anc.PolymorphicSiteFilter(samples=ingroup)
truth = anc.Grade.truth_at_focal(ts, ingroup_samples=ingroup, outgroup_samples=outgroup)
anc.Grade(res, truth, filter=keep)


118 sites: MAP recovery 98.3%; mean Brier = 0.031

## Saving the output

{meth}`ARGBasedInference.to_arg() <ancestree.inference.ARGBasedInference.to_arg>` writes a `.trees` file whose sites carry the MAP allele as their ancestral state and the posterior in their metadata (see {doc}`io`).


In [4]:
inf.to_arg("annotated.trees", posteriors=res);


INFO:ancestree.TskitWriter: Wrote 224 annotated sites to annotated.trees


In [5]:
anc.Reader("annotated.trees").head(3)


chrom   pos  alleles  AA       A       C       G       T
1       556  G/C      G   0.0000  0.0005  0.9995  0.0000
1       927  T/G      T   0.0000  0.0000  0.0000  1.0000
1      1031  T/A      T   0.0005  0.0000  0.0000  0.9995

## Misspecifying the mutation rate

:::{note}
{paramref}`mu <ancestree.inference.ARGBasedInference.mu>` must match the tree's time units: {mod}`tskit` reports branch lengths in {attr}`tskit.TreeSequence.time_units`, generations for {func}`msprime.sim_ancestry` simulations, so {class}`~ancestree.models.JC69` expects `mu` in per-site per-time-unit. Only the product `mu · branch_length` enters the kernel as expected substitutions per site, so a `mu` above the truth raises every transition probability and moves the posteriors toward the stationary distribution.

In the other direction, mu values below the truth barely shift either MAP or posterior: as `mu · t → 0` the kernel collapses to parsimony, and the relative ranking of root candidates becomes purely topological.
:::

Below the same ARG is rerun with `mu` a thousandfold too high and a thousandfold too low. Too high, the MAP is almost intact, being determined mostly by which subclade each tip belongs to in the local tree, but the posteriors degrade toward uniform, so the mean Brier score rises sharply while MAP recovery barely moves. Too low, the kernel is at its maximum-parsimony limit, which here scores as well as the true rate.

In [6]:
res_mu = {}
for mu in (5e-5, 5e-11):
    inf_mu = anc.Inference.from_arg(
        ts, anc.JC69(), mu=mu,
        ingroup_samples=ingroup, outgroup_samples=outgroup,
    )
    res_mu[mu] = list(inf_mu.infer())


INFO:ancestree.ARGBasedInference: Inferring the ancestral allele at 224 sites over 29 local trees from the ARG
INFO:ancestree.ARGBasedInference: Using 6 ingroup sample(s): i0, i1, i2, i3, i4, i5
INFO:ancestree.ARGBasedInference: Using 2 outgroup sample(s): o0, o1
ARGBasedInference: 100%|██████████| 29/29 [00:00<00:00, 1433.58 trees/s]
INFO:ancestree.ARGBasedInference: Focal node: reported at the ingroup_mrca; 4 tree(s) where the ingroup is not monophyletic, so its MRCA subtends outgroup tips
INFO:ancestree.ARGBasedInference: Inferring the ancestral allele at 224 sites over 29 local trees from the ARG
INFO:ancestree.ARGBasedInference: Using 6 ingroup sample(s): i0, i1, i2, i3, i4, i5
INFO:ancestree.ARGBasedInference: Using 2 outgroup sample(s): o0, o1
ARGBasedInference: 100%|██████████| 29/29 [00:00<00:00, 2485.64 trees/s]
INFO:ancestree.ARGBasedInference: Focal node: reported at the ingroup_mrca; 4 tree(s) where the ingroup is not monophyletic, so its MRCA subtends outgroup tips


In [7]:
print(f"{'mu':<22} {'MAP recovery':>13} {'mean Brier':>12}")
for label, r in (("5e-8 (true)", res),
                 ("5e-5 (1000× too high)", res_mu[5e-5]),
                 ("5e-11 (1000× too low)", res_mu[5e-11])):
    g = anc.Grade(r, truth, filter=keep)
    print(f"{label:<22} {g.map_recovery:>13.1%} {g.brier:>12.3f}")


mu                      MAP recovery   mean Brier
5e-8 (true)                    98.3%        0.031
5e-5 (1000× too high)          96.6%        0.173
5e-11 (1000× too low)          98.3%        0.031


In [8]:
g_true = anc.Grade(res, truth, filter=keep)
g_high, g_low = (anc.Grade(res_mu[mu], truth, filter=keep) for mu in (5e-5, 5e-11))
assert g_true.map_recovery > 0.95
assert g_high.brier > 3 * g_true.brier
assert g_high.map_recovery > g_true.map_recovery - 0.05
assert g_low.map_recovery >= g_true.map_recovery - 0.01
assert g_low.brier <= g_true.brier + 0.01


In [9]:
from pathlib import Path

Path("annotated.trees").unlink(missing_ok=True)
